# SiteGuard — PPE detector training (Colab)

Self-contained notebook: no repo clone needed. Trains `yolo11n` and `yolo11s` on the
SHWD-mirror dataset (`vodan37/yolo-helmethead`), using the exact same honest
group-disjoint pHash split logic as the local repo (same seed = same split, since the
split is a deterministic function of the (sorted) input files).

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

Steps: install deps → Kaggle auth → download & flatten data → honest split →
train both models → benchmark → export ONNX → zip results for download.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics onnx onnxruntime-gpu imagehash tqdm kaggle
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


## 2. Kaggle authentication

Paste a **fresh** API token (Kaggle → Settings → API → Create New Token). Do not reuse a
token that has ever been pasted into a chat or notebook you plan to share — regenerate it
first.

In [ ]:
import getpass, pathlib
token = getpass.getpass("Kaggle API token (KGAT_...): ")
pathlib.Path("/root/.kaggle").mkdir(exist_ok=True)
pathlib.Path("/root/.kaggle/access_token").write_text(token)
pathlib.Path("/root/.kaggle/access_token").chmod(0o600)
del token


## 3. Download the dataset

In [ ]:
!mkdir -p data/shwd
!kaggle datasets download -d vodan37/yolo-helmethead -p data/shwd --unzip
!find data/shwd -maxdepth 3 -type d


## 4. Pipeline scripts

Written to disk verbatim from the local repo (`scripts/prepare_shwd_raw.py`,
`scripts/make_splits.py`, `scripts/apply_splits.py`, `scripts/train.py`,
`scripts/evaluate.py`, `scripts/export_onnx.py`) so this reproduces the identical
pipeline, including the class remap (mirror ships `{0: head, 1: helmet}`, we use
`{0: helmet, 1: head}`) and the seed=1337 honest split.

In [ ]:
!mkdir -p scripts configs reports models


In [ ]:
%%writefile scripts/prepare_shwd_raw.py
"""Flatten the vodan37/yolo-helmethead Kaggle mirror into a single image/label
pool and remap its class indices to our convention.

This mirror ships pre-converted to YOLO format (not VOC XML as originally
assumed) and pre-split into train/valid/test -- but that split is a naive
random split over a dataset known to contain near-duplicates, which is
exactly the leakage Phase 2's honest pHash split exists to catch. So we
undo their split here and let make_splits.py + apply_splits.py redo it
group-disjointly.

Class remap: the mirror's helm.yaml declares names: ['head', 'helmet']
(0=head, 1=helmet). Our configs/data_shwd.yaml declares 0=helmet, 1=head,
so indices are swapped here to match.
"""
import argparse
from pathlib import Path

MIRROR_TO_OURS = {0: 1, 1: 0}  # mirror head->1, mirror helmet->0


def remap_label(src: Path, dst: Path) -> None:
    lines_out = []
    for line in src.read_text().splitlines():
        if not line.strip():
            continue
        cls_id, *coords = line.split()
        new_id = MIRROR_TO_OURS[int(cls_id)]
        lines_out.append(f"{new_id} {' '.join(coords)}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(lines_out))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mirror-root", type=Path, required=True,
                     help="path to .../helm/helm (contains images/, labels/)")
    ap.add_argument("--out-root", type=Path, default=Path("data/shwd_raw"))
    a = ap.parse_args()

    n_images = n_boxes = 0
    for split in ["train", "valid", "test"]:
        img_dir = a.mirror_root / "images" / split
        label_dir = a.mirror_root / "labels" / split
        for img in sorted(img_dir.glob("*.jpg")):
            label_src = label_dir / f"{img.stem}.txt"
            if not label_src.exists():
                continue
            dst_img = a.out_root / "images" / img.name
            dst_img.parent.mkdir(parents=True, exist_ok=True)
            if not dst_img.exists() and not dst_img.is_symlink():
                dst_img.symlink_to(img.resolve())
            dst_label = a.out_root / "labels" / f"{img.stem}.txt"
            remap_label(label_src, dst_label)
            n_images += 1
            n_boxes += len(dst_label.read_text().splitlines())

    print(f"Combined {n_images} images ({n_boxes} boxes) from train/valid/test "
          f"into {a.out_root} with classes remapped to 0=helmet, 1=head")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/make_splits.py
"""Bucket images by perceptual hash, then split group-disjointly."""
import argparse
import json
import random
from collections import defaultdict
from pathlib import Path

import imagehash
from PIL import Image
from tqdm import tqdm


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--img-dir", type=Path, required=True)
    ap.add_argument("--out", type=Path, default=Path("configs/splits.json"))
    ap.add_argument("--seed", type=int, default=1337)
    args = ap.parse_args()

    paths = sorted(p for p in args.img_dir.iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

    groups = defaultdict(list)
    for p in tqdm(paths, desc="hashing"):
        with Image.open(p) as im:
            groups[str(imagehash.phash(im.convert("RGB")))].append(p.name)

    dupes = sum(len(v) - 1 for v in groups.values())
    print(f"{len(paths)} images -> {len(groups)} groups "
          f"({dupes} duplicates, {100 * dupes / len(paths):.1f}%)")

    # Shuffle groups, not images. That is the whole point.
    keys = list(groups)
    random.Random(args.seed).shuffle(keys)

    n_train, n_val = int(0.70 * len(paths)), int(0.15 * len(paths))
    splits, count = {"train": [], "val": [], "test": []}, 0
    for k in keys:
        bucket = ("train" if count < n_train
                  else "val" if count < n_train + n_val else "test")
        splits[bucket].extend(groups[k])
        count += len(groups[k])

    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(splits, indent=2))
    for k, v in splits.items():
        print(f"  {k}: {len(v)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/apply_splits.py
"""Materialise configs/splits.json into the images/{split}/ and labels/{split}/
layout that data_shwd.yaml expects. Symlinks by default so re-running is cheap
and the dataset is never duplicated on disk.
"""
import argparse
import json
from pathlib import Path


def link_or_copy(src: Path, dst: Path, copy: bool) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if copy:
        dst.write_bytes(src.read_bytes())
    else:
        dst.symlink_to(src.resolve())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits", type=Path, default=Path("configs/splits.json"))
    ap.add_argument("--img-dir", type=Path, required=True)
    ap.add_argument("--label-dir", type=Path, required=True)
    ap.add_argument("--out-root", type=Path, required=True)
    ap.add_argument("--copy", action="store_true", help="copy instead of symlink")
    args = ap.parse_args()

    splits = json.loads(args.splits.read_text())
    for split, names in splits.items():
        missing_labels = 0
        for name in names:
            img_src = args.img_dir / name
            label_src = args.label_dir / f"{Path(name).stem}.txt"

            link_or_copy(img_src, args.out_root / "images" / split / name, args.copy)
            if label_src.exists():
                link_or_copy(label_src,
                             args.out_root / "labels" / split / f"{Path(name).stem}.txt",
                             args.copy)
            else:
                missing_labels += 1
        print(f"{split}: {len(names)} images, {missing_labels} missing labels")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train.py
import argparse
from ultralytics import YOLO

ap = argparse.ArgumentParser()
ap.add_argument("--model", default="yolo11s")
ap.add_argument("--data", default="configs/data_shwd.yaml")
ap.add_argument("--epochs", type=int, default=80)
ap.add_argument("--imgsz", type=int, default=640)
ap.add_argument("--batch", type=int, default=16)
a = ap.parse_args()

YOLO(f"{a.model}.pt").train(
    data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch,
    project="runs/ppe", name=a.model,
    seed=1337, deterministic=True, patience=20, cos_lr=True,
    # augmentation tuned for this domain
    hsv_v=0.5,          # worksites have brutal lighting variation
    degrees=5.0,        # heads are upright; big rotations are unrealistic
    scale=0.5,          # heads appear at wildly different distances
    fliplr=0.5,
    flipud=0.0,         # never vertical-flip a construction site
    close_mosaic=10,    # disable mosaic for the last 10 epochs
    copy_paste=0.1,     # cheap help for the minority helmet class
)


In [ ]:
%%writefile scripts/evaluate.py
import argparse, json, time
from pathlib import Path

import numpy as np
import torch
from ultralytics import YOLO


def latency(model, imgsz: int, device: str, warmup=20, runs=100) -> dict:
    dummy = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)
    for _ in range(warmup):                      # never benchmark a cold model
        model.predict(dummy, imgsz=imgsz, device=device, verbose=False)

    times = []
    for _ in range(runs):
        if device != "cpu":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.predict(dummy, imgsz=imgsz, device=device, verbose=False)
        if device != "cpu":
            torch.cuda.synchronize()             # without this, GPU numbers are fiction
        times.append((time.perf_counter() - t0) * 1000)

    t = np.array(times)
    return {"p50_ms": round(float(np.percentile(t, 50)), 2),
            "fps": round(1000 / float(t.mean()), 1)}


def evaluate(weights: Path, data: str, imgsz: int, device: str) -> dict:
    model = YOLO(str(weights))
    m = model.val(data=data, split="test", imgsz=imgsz, device=device, verbose=False)

    row = {"model": weights.parent.parent.name,
           "size_MB": round(weights.stat().st_size / 1e6, 1),
           "mAP50": round(float(m.box.map50), 4),
           "mAP50_95": round(float(m.box.map), 4)}
    for i, ap in enumerate(m.box.ap50):
        row[f"AP50_{model.names[i]}"] = round(float(ap), 4)

    row |= {f"gpu_{k}": v for k, v in latency(model, imgsz, device).items()}
    row |= {f"cpu_{k}": v for k, v in latency(model, imgsz, "cpu").items()}
    return row


def to_md(rows: list[dict]) -> str:
    cols = list(rows[0])
    return "\n".join([
        "| " + " | ".join(cols) + " |",
        "|" + "|".join("---" for _ in cols) + "|",
        *["| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows],
    ])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs-dir", type=Path, default=Path("runs/ppe"))
    ap.add_argument("--data", default="configs/data_shwd.yaml")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--device", default="0")
    a = ap.parse_args()

    rows = [evaluate(w, a.data, a.imgsz, a.device)
            for w in sorted(a.runs_dir.glob("*/weights/best.pt"))]
    Path("reports").mkdir(exist_ok=True)
    Path("reports/benchmark.json").write_text(json.dumps(rows, indent=2))
    Path("reports/benchmark.md").write_text(to_md(rows))
    print(to_md(rows))


In [ ]:
%%writefile scripts/export_onnx.py
"""Export the trained yolo11s checkpoint to ONNX FP32."""
import argparse
from pathlib import Path
from ultralytics import YOLO

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", type=Path, default=Path("runs/ppe/yolo11s/weights/best.pt"))
    ap.add_argument("--imgsz", type=int, default=640)
    a = ap.parse_args()

    YOLO(str(a.weights)).export(
        format="onnx", imgsz=a.imgsz, opset=13, simplify=True, dynamic=False,
    )


## 5. Flatten + remap, then honest pHash split

In [ ]:
!python scripts/prepare_shwd_raw.py --mirror-root data/shwd/helm/helm --out-root data/shwd_raw
!python scripts/make_splits.py --img-dir data/shwd_raw/images --out configs/splits.json --seed 1337
!python scripts/apply_splits.py --splits configs/splits.json \
    --img-dir data/shwd_raw/images --label-dir data/shwd_raw/labels \
    --out-root data/shwd_final


**Cross-check against the local run:** the local (CPU-only) machine already ran this on the
same Kaggle mirror and got:

```
22789 images -> 21555 groups (1234 duplicates, 5.4%)
  train: 15952
  val: 3418
  test: 3419
```

If the numbers above don't match, the Kaggle mirror content has changed since — flag it before trusting the rest of the run.

In [ ]:
import pathlib
data_yaml = f'''path: {pathlib.Path("data/shwd_final").resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: helmet
  1: head
'''
pathlib.Path("configs").mkdir(exist_ok=True)
pathlib.Path("configs/data_shwd.yaml").write_text(data_yaml)
print(data_yaml)


## 6. Train both models

~80 epochs each at imgsz=640. On a T4 this is roughly 2-4 hours per model depending on
queue/throttling — budget for a Colab Pro / background-execution session if on the free
tier, since idle disconnects will kill an in-progress run. Runs checkpoint to
`runs/ppe/<model>/weights/` as they go; if disconnected, `YOLO("runs/ppe/<model>/weights/last.pt").train(resume=True)` picks back up.

In [ ]:
!python scripts/train.py --model yolo11n --epochs 80 --imgsz 640 --batch 32


In [ ]:
!python scripts/train.py --model yolo11s --epochs 80 --imgsz 640 --batch 32


## 7. Checkpoint to Drive (recommended)

Training runs live in ephemeral Colab storage — copy weights out immediately so a runtime
recycle doesn't lose hours of training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/siteguard_runs
!cp -r runs/ppe /content/drive/MyDrive/siteguard_runs/
print("Copied to /content/drive/MyDrive/siteguard_runs/ppe")


## 8. Benchmark (accuracy + latency, GPU and CPU)

In [ ]:
!python scripts/evaluate.py --runs-dir runs/ppe --data configs/data_shwd.yaml --device 0


## 9. Export the best model (yolo11s) to ONNX

In [ ]:
!python scripts/export_onnx.py --weights runs/ppe/yolo11s/weights/best.pt --imgsz 640


## 10. Package everything for download back into the local repo

In [ ]:
!mkdir -p out/models
!cp runs/ppe/yolo11n/weights/best.pt out/models/yolo11n_best.pt
!cp runs/ppe/yolo11s/weights/best.pt out/models/yolo11s_best.pt
!cp runs/ppe/yolo11s/weights/best.onnx out/models/best.onnx
!cp -r reports out/
!cp configs/splits.json out/
!cd out && zip -r ../siteguard_trained.zip . && cd ..
from google.colab import files
files.download('siteguard_trained.zip')


## 11. After downloading

Unzip `siteguard_trained.zip` into the local repo root — it recreates `models/`,
`reports/`, and `configs/splits.json`. Then locally:

```bash
cp models/best.onnx models/best.onnx   # already in place after unzip
make serve                             # or: docker compose up
```

Copy the `reports/benchmark.md` table and the per-class AP numbers into the main
README's Benchmark section (see Definition of Done in README.md).